# SBS inline validation: OMTPL Retail Classical FREQ

Ноутбук запускает тесты SBS `M0`-`M5` по отдельным ячейкам и показывает результат каждого теста inline прямо под ячейкой.

Последняя ячейка собирает единый HTML-отчет из уже рассчитанных inline-результатов. Если какие-то тесты не были запущены выше, финальная сборка может досчитать их без отображения.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_FILENAME = "OMTPL_Retail_Classical_FREQ_Inference_db_ver01.csv"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / DATA_FILENAME
REPORTS_DIR = PROJECT_ROOT / "reports" / "omtpl_retail_classical_freq_inline"

if not DATA_PATH.exists():
    candidates = sorted(PROJECT_ROOT.rglob(DATA_FILENAME))
    if candidates:
        DATA_PATH = candidates[0]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"REPORTS_DIR: {REPORTS_DIR}")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Не найден {DATA_FILENAME}. Положите файл в data/raw/ или рядом с проектом, "
        "либо поправьте DATA_PATH в этой ячейке."
    )

In [ ]:
EXPECTED_SERVICE_COLUMNS = [
    "UNIQUE_KEY",
    "DB_SPLIT",
    "TIME_TREND_ID_START_QUARTER",
    "TIME_TREND_SEAS_REPORT_YEAR_QUARTER",
    "Exposure",
    "ClaimsCount",
    "GLM_FREQ",
    "GLM_FREQ_weighted",
    "GLM_MODEL_BINQ10",
    "GLM_MODEL_weighted_BINQ10",
]

EXPECTED_RF_LEVEL_COLUMNS = """
COMB_DTP_COMB_PRED_KBM
COMB_ind_ldu_names
COMB_OWNER_IN_LDU_LEVEL_COEFF
COMB_YEARS_WO_DTP_KBM_GIBDD
DTPHIST_YEARS_OWNED
HT_MKT_DTP_PROB_PROM2026_W250
MKT_COND_PREV1_POLICY_ADDENDUM_IND
MKT_CURR_Customer_walk_2Ys_Switch_Break
MKT_DTP_SCORE_COMBINED_PROB_W250
MKT_DTP_SCORE_MODULES_PRSN_PROB_W100
PAO_RATING_FACTORNAME_1
PAO_RATING_FACTORNAME_2_W100_FREQ
PEOPLE_AGE_WORST
PEOPLE_correct_kbm
PEOPLE_DRIVERS_MULTIDRIVE
PEOPLE_EXP_WORST
PEOPLE_GENDER_WORST
PEOPLE_KBM_WORST
PEOPLE_OWNER_AGE
PEOPLE_OWNER_CHANGED_FLAG
PEOPLE_OWNER_IN_LDU_FLAG
PEOPLE_WORST_DRIVING_LICENCE_AGE
POL_days_between_prev_curr_pol
REG_FED_DISTRICT
REG_FO_REGION
REGION_Adm08_Distance_FederalHighWays_wo_CAPs_km_gr
REGION_Adm08_Distance_Nearest_City100kpl_km_gr
REGION_MKT_ENCODING_Adm08_a6_rnd
SBS_PAO_SEGMENT
SBS_PAO_TYPE
SBS_PROD_CUSTOMER_SEGMENT
SEAS_NERAVNOM_RISK_MONTHS
SEAS_REPORT_YEAR_QUARTER
VEHICLE_AGE_FINAL
VEHICLE_Brand_Country_Fin
VEHICLE_CAR_PURPOSE
VEHICLE_CATALOG_PRICE_Avg_50
VEHICLE_CATALOG_SHARE_INSURANT_Female_BrandModel_v1
VEHICLE_CATEGORY_TS
VEHICLE_MAKE_MODEL
VEHICLE_OWNERSHIP_multiplicity
VEHICLE_PMW_to_Power_LPV_Round
VEHICLE_Power_HP_LPV
VEHICLE_RAMR_BODY_TYPE_LPV
VEHICLE_RAMR_FUEL_TYPE
VEHICLE_RAMR_LEASING_STATUS
VEHICLE_RAMR_WHEEL_TYPE
VEHICLE_RTA_CNT_BEFOREOWNER_ALL_TIME_DAMAGE_CODES_MAX
VEHICLE_RTA_CNT_BEFOREOWNER_Time_from_last_RTA_m
VEHICLE_RTA_CNT_OWNER_1YEAR_TOTAL_CLMS
VEHICLE_RTA_CNT_OWNER_3YEAR_TIME_EVENING_CNT
VEHICLE_RTA_CNT_OWNER_ALL_TIME_COLLISION_CNT
VEHICLE_RTA_CNT_OWNER_ALL_TIME_OVERTURN_CNT
VEHICLE_RTA_CNT_OWNER_freq_5Y_total_clms_Q27
VEHICLE_SIZE_CLASS_02
VEHICLE_TAXI_IS_TAXI_OWNER_BEFORE_POLICY
VEHICLE_ts_period
VEHICLE_Weight_PMW
""".split()

EXPECTED_SCORING_COLUMNS = """
COMB_DTP_COMB_PRED_KBM_SCORING
COMB_ind_ldu_names_SCORING
COMB_OWNER_IN_LDU_LEVEL_COEFF_SCORING
COMB_YEARS_WO_DTP_KBM_GIBDD_SCORING
DTPHIST_YEARS_OWNED_SCORING
HT_MKT_DTP_PROB_PROM2026_W250_SCORING
INTERCEPT
MKT_COND_PREV1_POLICY_ADDENDUM_IND_SCORING
MKT_CURR_Customer_walk_2Ys_Switch_Break_SCORING
MKT_DTP_SCORE_COMBINED_PROB_W250_SCORING
MKT_DTP_SCORE_MODULES_PRSN_PROB_W100_SCORING
PAO_RATING_FACTORNAME_1_SCORING
PAO_RATING_FACTORNAME_2_W100_FREQ_SCORING
PEOPLE_AGE_WORST_SCORING
PEOPLE_correct_kbm_SCORING
PEOPLE_DRIVERS_MULTIDRIVE_SCORING
PEOPLE_EXP_WORST_SCORING
PEOPLE_EXP_WORST_x_PEOPLE_GENDER_WORST_SCORING
PEOPLE_GENDER_WORST_x_PEOPLE_KBM_WORST_SCORING
PEOPLE_KBM_WORST_SCORING
PEOPLE_OWNER_AGE_x_PEOPLE_OWNER_IN_LDU_FLAG_SCORING
PEOPLE_OWNER_CHANGED_FLAG_SCORING
PEOPLE_WORST_DRIVING_LICENCE_AGE_SCORING
POL_days_between_prev_curr_pol_SCORING
REG_FED_DISTRICT_x_REGION_Adm08_Distance_FederalHighWays_wo_CAPs_km_gr_SCORING
REG_FO_REGION_SCORING
REGION_Adm08_Distance_Nearest_City100kpl_km_gr_SCORING
REGION_MKT_ENCODING_Adm08_a6_rnd_SCORING
SBS_PAO_SEGMENT_SCORING
SBS_PAO_TYPE_SCORING
SBS_PROD_CUSTOMER_SEGMENT_SCORING
SEAS_NERAVNOM_RISK_MONTHS_SCORING
SEAS_REPORT_YEAR_QUARTER_SCORING
VEHICLE_AGE_FINAL_SCORING
VEHICLE_Brand_Country_Fin_SCORING
VEHICLE_CAR_PURPOSE_SCORING
VEHICLE_CATALOG_PRICE_Avg_50_SCORING
VEHICLE_CATALOG_SHARE_INSURANT_Female_BrandModel_v1_SCORING
VEHICLE_CATEGORY_TS_SCORING
VEHICLE_MAKE_MODEL_SCORING
VEHICLE_OWNERSHIP_multiplicity_SCORING
VEHICLE_PMW_to_Power_LPV_Round_SCORING
VEHICLE_Power_HP_LPV_SCORING
VEHICLE_Power_HP_LPV_x_PEOPLE_DRIVERS_MULTIDRIVE_SCORING
VEHICLE_RAMR_BODY_TYPE_LPV_SCORING
VEHICLE_RAMR_FUEL_TYPE_SCORING
VEHICLE_RAMR_LEASING_STATUS_SCORING
VEHICLE_RAMR_WHEEL_TYPE_SCORING
VEHICLE_RTA_CNT_BEFOREOWNER_ALL_TIME_DAMAGE_CODES_MAX_SCORING
VEHICLE_RTA_CNT_BEFOREOWNER_Time_from_last_RTA_m_SCORING
VEHICLE_RTA_CNT_OWNER_1YEAR_TOTAL_CLMS_SCORING
VEHICLE_RTA_CNT_OWNER_3YEAR_TIME_EVENING_CNT_SCORING
VEHICLE_RTA_CNT_OWNER_ALL_TIME_COLLISION_CNT_SCORING
VEHICLE_RTA_CNT_OWNER_ALL_TIME_OVERTURN_CNT_SCORING
VEHICLE_RTA_CNT_OWNER_freq_5Y_total_clms_Q27_SCORING
VEHICLE_SIZE_CLASS_02_SCORING
VEHICLE_TAXI_IS_TAXI_OWNER_BEFORE_POLICY_SCORING
VEHICLE_ts_period_SCORING
VEHICLE_Weight_PMW_SCORING
""".split()

EXPECTED_VALUE_COLUMNS = """
COMB_DTP_COMB_PRED_KBM_VALUE
COMB_ind_ldu_names_VALUE
COMB_OWNER_IN_LDU_LEVEL_COEFF_VALUE
COMB_YEARS_WO_DTP_KBM_GIBDD_VALUE
DTPHIST_YEARS_OWNED_VALUE
HT_MKT_DTP_PROB_PROM2026_W250_VALUE
INTERCEPT_VALUE
MKT_COND_PREV1_POLICY_ADDENDUM_IND_VALUE
MKT_CURR_Customer_walk_2Ys_Switch_Break_VALUE
MKT_DTP_SCORE_COMBINED_PROB_W250_VALUE
MKT_DTP_SCORE_MODULES_PRSN_PROB_W100_VALUE
PAO_RATING_FACTORNAME_1_VALUE
PAO_RATING_FACTORNAME_2_W100_FREQ_VALUE
PEOPLE_AGE_WORST_VALUE
PEOPLE_correct_kbm_VALUE
PEOPLE_DRIVERS_MULTIDRIVE_VALUE
PEOPLE_EXP_WORST_VALUE
PEOPLE_EXP_WORST_x_PEOPLE_GENDER_WORST_VALUE
PEOPLE_GENDER_WORST_x_PEOPLE_KBM_WORST_VALUE
PEOPLE_KBM_WORST_VALUE
PEOPLE_OWNER_AGE_x_PEOPLE_OWNER_IN_LDU_FLAG_VALUE
PEOPLE_OWNER_CHANGED_FLAG_VALUE
PEOPLE_WORST_DRIVING_LICENCE_AGE_VALUE
POL_days_between_prev_curr_pol_VALUE
REG_FED_DISTRICT_x_REGION_Adm08_Distance_FederalHighWays_wo_CAPs_km_gr_VALUE
REG_FO_REGION_VALUE
REGION_Adm08_Distance_Nearest_City100kpl_km_gr_VALUE
REGION_MKT_ENCODING_Adm08_a6_rnd_VALUE
SBS_PAO_SEGMENT_VALUE
SBS_PAO_TYPE_VALUE
SBS_PROD_CUSTOMER_SEGMENT_VALUE
SEAS_NERAVNOM_RISK_MONTHS_VALUE
SEAS_REPORT_YEAR_QUARTER_VALUE
VEHICLE_AGE_FINAL_VALUE
VEHICLE_Brand_Country_Fin_VALUE
VEHICLE_CAR_PURPOSE_VALUE
VEHICLE_CATALOG_PRICE_Avg_50_VALUE
VEHICLE_CATALOG_SHARE_INSURANT_Female_BrandModel_v1_VALUE
VEHICLE_CATEGORY_TS_VALUE
VEHICLE_MAKE_MODEL_VALUE
VEHICLE_OWNERSHIP_multiplicity_VALUE
VEHICLE_PMW_to_Power_LPV_Round_VALUE
VEHICLE_Power_HP_LPV_VALUE
VEHICLE_Power_HP_LPV_x_PEOPLE_DRIVERS_MULTIDRIVE_VALUE
VEHICLE_RAMR_BODY_TYPE_LPV_VALUE
VEHICLE_RAMR_FUEL_TYPE_VALUE
VEHICLE_RAMR_LEASING_STATUS_VALUE
VEHICLE_RAMR_WHEEL_TYPE_VALUE
VEHICLE_RTA_CNT_BEFOREOWNER_ALL_TIME_DAMAGE_CODES_MAX_VALUE
VEHICLE_RTA_CNT_BEFOREOWNER_Time_from_last_RTA_m_VALUE
VEHICLE_RTA_CNT_OWNER_1YEAR_TOTAL_CLMS_VALUE
VEHICLE_RTA_CNT_OWNER_3YEAR_TIME_EVENING_CNT_VALUE
VEHICLE_RTA_CNT_OWNER_ALL_TIME_COLLISION_CNT_VALUE
VEHICLE_RTA_CNT_OWNER_ALL_TIME_OVERTURN_CNT_VALUE
VEHICLE_RTA_CNT_OWNER_freq_5Y_total_clms_Q27_VALUE
VEHICLE_SIZE_CLASS_02_VALUE
VEHICLE_TAXI_IS_TAXI_OWNER_BEFORE_POLICY_VALUE
VEHICLE_ts_period_VALUE
VEHICLE_Weight_PMW_VALUE
""".split()

EXPECTED_INTERCEPT_COLUMNS = ["INTERCEPT", "INTERCEPT_VALUE"]
EXPECTED_RAW_SCHEMA_COLUMNS = list(dict.fromkeys([
    *EXPECTED_SERVICE_COLUMNS,
    *EXPECTED_RF_LEVEL_COLUMNS,
    *EXPECTED_SCORING_COLUMNS,
    *EXPECTED_VALUE_COLUMNS,
]))
EXPECTED_FACTOR_COLUMNS = list(dict.fromkeys([
    *EXPECTED_RF_LEVEL_COLUMNS,
    *[c for c in EXPECTED_SCORING_COLUMNS if c not in EXPECTED_INTERCEPT_COLUMNS],
    *[c for c in EXPECTED_VALUE_COLUMNS if c not in EXPECTED_INTERCEPT_COLUMNS],
]))

def columns_present(df, expected: list[str]) -> list[str]:
    by_lower = {str(c).lower(): c for c in df.columns}
    out = []
    for column in expected:
        actual = column if column in df.columns else by_lower.get(column.lower())
        if actual is not None:
            out.append(actual)
    return list(dict.fromkeys(out))

def columns_missing(df, expected: list[str]) -> list[str]:
    present_lower = {str(c).lower() for c in df.columns}
    return [column for column in expected if column.lower() not in present_lower]

print("Expected raw columns from pasted schema:", len(EXPECTED_RAW_SCHEMA_COLUMNS))
print("Expected RF-level columns:", len(EXPECTED_RF_LEVEL_COLUMNS))
print("Expected scoring columns including INTERCEPT:", len(EXPECTED_SCORING_COLUMNS))
print("Expected value columns including INTERCEPT_VALUE:", len(EXPECTED_VALUE_COLUMNS))
print("Expected factor columns for SBS excluding service/intercepts:", len(EXPECTED_FACTOR_COLUMNS))

## Загрузка данных

In [ ]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    attempts = [
        {"sep": None, "engine": "python"},
        {"sep": ";", "encoding": "cp1251", "decimal": ","},
        {"sep": ";", "encoding": "utf-8", "decimal": ","},
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ",", "encoding": "cp1251"},
    ]
    errors = []
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if df.shape[1] > 1:
                print(f"read_csv kwargs: {kwargs}")
                return df
        except Exception as exc:
            errors.append(f"{kwargs}: {exc}")
    raise RuntimeError("Не удалось прочитать CSV:\n" + "\n".join(errors))

data = read_csv_safely(DATA_PATH)
data.columns = data.columns.astype(str).str.strip()
print(f"Shape: {data.shape[0]:,} rows x {data.shape[1]:,} columns")
display(data.head())
display(pd.DataFrame({"column": data.columns, "dtype": data.dtypes.astype(str).values}).head(120))

## Настройка колонок



In [ ]:
def require_column(df: pd.DataFrame, column: str, required: bool = True) -> str | None:
    by_lower = {str(c).lower(): c for c in df.columns}
    actual = column if column in df.columns else by_lower.get(column.lower())
    if actual is not None:
        return actual
    if required:
        raise KeyError(f"Не найдена обязательная колонка схемы: {column}")
    return None

# Явный контракт схемы для OMTPL Retail Classical FREQ.
# Здесь нет выбора из кандидатов: если имя поменялось, лучше поправить контракт руками.
SCORE_COLUMN = require_column(data, "GLM_FREQ")
WEIGHTED_SCORE_COLUMN = require_column(data, "GLM_FREQ_weighted", required=False)
WEIGHT_COLUMN = require_column(data, "Exposure")
SAMPLE_COLUMN = require_column(data, "DB_SPLIT")
DATE_COLUMN = require_column(data, "TIME_TREND_SEAS_REPORT_YEAR_QUARTER")
TARGET_COLUMN = require_column(data, "ClaimsCount")
ID_COLUMN = require_column(data, "UNIQUE_KEY")

TIME_FACTOR_SOURCE_COLUMNS = [
    c for c in [
        require_column(data, "TIME_TREND_ID_START_QUARTER", required=False),
        require_column(data, "TIME_TREND_SEAS_REPORT_YEAR_QUARTER", required=False),
    ]
    if c is not None
]
PREDICTION_BIN_COLUMNS = [
    c for c in [
        require_column(data, "GLM_MODEL_BINQ10", required=False),
        require_column(data, "GLM_MODEL_weighted_BINQ10", required=False),
    ]
    if c is not None
]
INTERCEPT_COLUMNS = [
    c for c in [require_column(data, column, required=False) for column in EXPECTED_INTERCEPT_COLUMNS]
    if c is not None
]

selected_columns = {
    "score_column": SCORE_COLUMN,
    "weighted_score_diagnostic": WEIGHTED_SCORE_COLUMN,
    "weight_column": WEIGHT_COLUMN,
    "sample_column": SAMPLE_COLUMN,
    "date_column": DATE_COLUMN,
    "target_column": TARGET_COLUMN,
    "id_column": ID_COLUMN,
    "time_columns_not_used_as_features": TIME_FACTOR_SOURCE_COLUMNS,
    "prediction_bin_columns_excluded_from_features": PREDICTION_BIN_COLUMNS,
    "intercept_columns_excluded_from_features": INTERCEPT_COLUMNS,
}
selected_columns

## Подготовка данных

In [ ]:
def to_numeric_strict(series: pd.Series, column: str) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        out = pd.to_numeric(series, errors="coerce")
    else:
        normalized = (
            series.astype(str)
            .str.strip()
            .str.replace(" ", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        out = pd.to_numeric(normalized, errors="coerce")
    bad = out.isna() & series.notna() & ~series.astype(str).str.strip().isin(["", "nan", "None"])
    if bad.any():
        examples = series[bad].astype(str).head(10).tolist()
        raise ValueError(f"Колонка {column} не приводится к numeric, примеры: {examples}")
    return out

def parse_report_date(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series, errors="raise")
    for fmt in ("%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d"):
        parsed = pd.to_datetime(series, format=fmt, errors="coerce")
        if parsed.notna().mean() >= 0.95:
            return parsed
    return pd.to_datetime(series, errors="raise", dayfirst=True)

final_df = data.copy()

final_df[DATE_COLUMN] = parse_report_date(final_df[DATE_COLUMN])
final_df[SAMPLE_COLUMN] = final_df[SAMPLE_COLUMN].astype(str).str.strip().str.upper()
final_df[ID_COLUMN] = final_df[ID_COLUMN].astype(str)

for column in [TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN]:
    final_df[column] = to_numeric_strict(final_df[column], column)
    if final_df[column].isna().any():
        raise ValueError(f"В обязательной колонке {column} есть пропуски: {int(final_df[column].isna().sum())}")

if (final_df[WEIGHT_COLUMN] <= 0).any():
    bad_n = int((final_df[WEIGHT_COLUMN] <= 0).sum())
    raise ValueError(f"В колонке экспозиции {WEIGHT_COLUMN} есть неположительные значения: {bad_n}")

if WEIGHTED_SCORE_COLUMN is not None:
    final_df[WEIGHTED_SCORE_COLUMN] = to_numeric_strict(final_df[WEIGHTED_SCORE_COLUMN], WEIGHTED_SCORE_COLUMN)
    weighted_expected = final_df[SCORE_COLUMN] * final_df[WEIGHT_COLUMN]
    max_abs_diff = (final_df[WEIGHTED_SCORE_COLUMN] - weighted_expected).abs().max()
    print(f"Weighted prediction check: max |{WEIGHTED_SCORE_COLUMN} - {SCORE_COLUMN} * {WEIGHT_COLUMN}| = {max_abs_diff:.6g}")

split_summary = final_df[SAMPLE_COLUMN].value_counts(dropna=False).rename_axis(SAMPLE_COLUMN).reset_index(name="rows")
date_summary = pd.DataFrame({
    "metric": ["min_date", "max_date", "missing_dates"],
    "value": [final_df[DATE_COLUMN].min(), final_df[DATE_COLUMN].max(), final_df[DATE_COLUMN].isna().sum()],
})

display(split_summary)
display(date_summary)
display(final_df[[TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN]].describe().T)

## Список факторов



In [ ]:
service_columns = {
    SCORE_COLUMN,
    WEIGHTED_SCORE_COLUMN,
    WEIGHT_COLUMN,
    SAMPLE_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    ID_COLUMN,
    *TIME_FACTOR_SOURCE_COLUMNS,
    *PREDICTION_BIN_COLUMNS,
    *INTERCEPT_COLUMNS,
}
service_columns = {c for c in service_columns if c is not None}

# Явное разделение факторов по переданному описанию схемы:
# - GLM_RF_levels -> categorical_features
# - *_SCORING и *_VALUE без интерсептов -> numeric_features
numeric_feature_contract = list(dict.fromkeys([
    *[c for c in EXPECTED_SCORING_COLUMNS if c not in EXPECTED_INTERCEPT_COLUMNS],
    *[c for c in EXPECTED_VALUE_COLUMNS if c not in EXPECTED_INTERCEPT_COLUMNS],
]))
categorical_feature_contract = list(dict.fromkeys(EXPECTED_RF_LEVEL_COLUMNS))

numeric_features = [c for c in columns_present(final_df, numeric_feature_contract) if c not in service_columns]
categorical_features = [c for c in columns_present(final_df, categorical_feature_contract) if c not in service_columns]

missing_numeric_features = columns_missing(final_df, numeric_feature_contract)
missing_categorical_features = columns_missing(final_df, categorical_feature_contract)
feature_overlap = sorted(set(numeric_features).intersection(categorical_features))
feature_service_overlap = sorted(service_columns.intersection(numeric_features + categorical_features))

schema_audit = pd.DataFrame([
    {
        "group": group,
        "expected": len(expected),
        "present": len(columns_present(final_df, expected)),
        "missing": len(columns_missing(final_df, expected)),
        "missing_columns": columns_missing(final_df, expected),
    }
    for group, expected in {
        "service": EXPECTED_SERVICE_COLUMNS,
        "rf_level_categorical": categorical_feature_contract,
        "scoring_numeric_ex_intercept": [c for c in EXPECTED_SCORING_COLUMNS if c not in EXPECTED_INTERCEPT_COLUMNS],
        "value_numeric_ex_intercept": [c for c in EXPECTED_VALUE_COLUMNS if c not in EXPECTED_INTERCEPT_COLUMNS],
    }.items()
])

display(schema_audit)

if missing_numeric_features or missing_categorical_features:
    raise KeyError(
        "В данных отсутствуют ожидаемые факторные колонки. "
        f"numeric_missing={missing_numeric_features}; "
        f"categorical_missing={missing_categorical_features}"
    )
if feature_overlap:
    raise ValueError(f"Факторы не должны одновременно быть numeric и categorical: {feature_overlap}")
if feature_service_overlap:
    raise ValueError(f"Служебные колонки попали в список факторов: {feature_service_overlap}")
if not numeric_features and not categorical_features:
    raise ValueError("Список факторов пуст. Проверьте EXPECTED_* списки схемы.")

feature_missing = []
for column in numeric_features:
    final_df[column] = to_numeric_strict(final_df[column], column)
    missing = int(final_df[column].isna().sum())
    if missing:
        feature_missing.append({"feature": column, "missing_filled_with_zero": missing})
        final_df[column] = final_df[column].fillna(0.0)

for column in categorical_features:
    final_df[column] = final_df[column].where(final_df[column].notna(), "__MISSING__").astype(str)

unexpected_raw_columns = sorted(
    c for c in final_df.columns
    if c not in columns_present(final_df, EXPECTED_RAW_SCHEMA_COLUMNS)
)

print("Feature contract: explicit schema lists")
print(f"Total columns in file: {len(final_df.columns)}")
print(f"Service columns: {len(service_columns)}")
print(f"Numeric features used: {len(numeric_features)}")
print(f"Categorical features used: {len(categorical_features)}")
print(f"Total SBS features used: {len(numeric_features) + len(categorical_features)}")
print(f"Expected raw columns from pasted schema: {len(EXPECTED_RAW_SCHEMA_COLUMNS)}")
print(f"Expected raw columns present: {len(columns_present(final_df, EXPECTED_RAW_SCHEMA_COLUMNS))}")
print(f"Delta vs explicit schema factors: {len(numeric_features) + len(categorical_features) - len(EXPECTED_FACTOR_COLUMNS):+d}")

if feature_missing:
    print(f"WARNING: пропуски в {len(feature_missing)} numeric-факторах заполнены 0.0 для стабильного запуска тестов SBS.")
    display(pd.DataFrame(feature_missing).head(80))

if unexpected_raw_columns:
    print(f"WARNING: найдены {len(unexpected_raw_columns)} колонок вне переданного описания; они не включены в факторы.")
    display(pd.DataFrame({"unexpected_raw_columns": unexpected_raw_columns}).head(120))

display(pd.DataFrame({"excluded_from_features": sorted(service_columns)}))
display(pd.DataFrame({
    "numeric_features": pd.Series(numeric_features),
    "categorical_features": pd.Series(categorical_features),
}).head(120))

## Конфиг 

In [ ]:
config = {
    "model_mode": "frequency",
    "validation_mode": "oos",
    "columns": {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "score_column": SCORE_COLUMN,
        "prediction_column": None,
        "date_column": DATE_COLUMN,
        "target_column": TARGET_COLUMN,
        "sample_column": SAMPLE_COLUMN,
        "id_column": ID_COLUMN,
        "weight_column": WEIGHT_COLUMN,
    },
}

config

In [ ]:
from sbs_evaluation_tool.core.config_validation import validate_config

validated_config = validate_config(config, final_df)
print(validated_config)
print(f"Numeric features: {len(validated_config.columns.numeric_features)}")
print(f"Categorical features: {len(validated_config.columns.categorical_features)}")

## Inline-запуск тестов

Каждая следующая ячейка запускает один тест 


In [ ]:
from html import escape
from pathlib import Path
from datetime import datetime

from IPython.display import HTML, FileLink, display

from sbs_evaluation_tool.core.base_test import ModelReportBuilder, TestGroupBuilder
from sbs_evaluation_tool.tests.psi_calculator import PSICalculatorExtended
from sbs_evaluation_tool.tests.m0_data_quality import (
    M01_DataQualityOverviewTest,
    M04_MonthlyDistributionsTest,
    M05_ModeAndMissingByPeriodTest,
)
from sbs_evaluation_tool.tests.m1_distribution_tests import (
    M11_TrainVsGenpopPSITest,
    M12_OOSVsGenpopPSITest,
    M13_MaxDateRecencyTest,
)
from sbs_evaluation_tool.tests.m2_gini_tests import (
    M21_GiniTest,
    M22_GiniFactorsTest,
    M23_GiniDynamicsTest,
    M24_GiniUpliftTest,
)
from sbs_evaluation_tool.tests.m3_multicollinearity import (
    M31_LikelihoodRatioTest,
    M32_MulticollinearityTest,
)
from sbs_evaluation_tool.tests.m4_model_performance import (
    M41_TargetRateTest,
    M42_CalibrationCurveByPredictBinsLn,
    M43_CalibrationCurveByDatesLn,
)
from sbs_evaluation_tool.tests.m5_model_stability import (
    M51_ModelGiniStabilityTest,
    M52_FactorGiniStabilityTest,
    M53_ScorePSITest,
    M54_FactorPSITest,
)

psi_calc = PSICalculatorExtended()
common_params = {"df": final_df, "config": validated_config}

TEST_FACTORIES = {
    "M 0.1": M01_DataQualityOverviewTest,
    "M 0.4": M04_MonthlyDistributionsTest,
    "M 0.5": M05_ModeAndMissingByPeriodTest,
    "M 1.1": lambda: M11_TrainVsGenpopPSITest(psi_calc=psi_calc),
    "M 1.2": lambda: M12_OOSVsGenpopPSITest(psi_calc=psi_calc),
    "M 1.3": M13_MaxDateRecencyTest,
    "M 2.1": M21_GiniTest,
    "M 2.2": M22_GiniFactorsTest,
    "M 2.3": M23_GiniDynamicsTest,
    "M 2.4": M24_GiniUpliftTest,
    "M 3.1": M31_LikelihoodRatioTest,
    "M 3.2": M32_MulticollinearityTest,
    "M 4.1": M41_TargetRateTest,
    "M 4.2": M42_CalibrationCurveByPredictBinsLn,
    "M 4.3": M43_CalibrationCurveByDatesLn,
    "M 5.1": M51_ModelGiniStabilityTest,
    "M 5.2": M52_FactorGiniStabilityTest,
    "M 5.3": lambda: M53_ScorePSITest(psi_calc=psi_calc),
    "M 5.4": lambda: M54_FactorPSITest(psi_calc=psi_calc),
}

TEST_PARAMS = {code: dict(common_params) for code in TEST_FACTORIES}
TEST_PARAMS["M 4.2"] = {**common_params, "num_groups": 20}

TEST_GROUPS = [
    ("M0", "M0. Data Quality", ["M 0.1", "M 0.4", "M 0.5"]),
    ("M1", "M1. Distribution", ["M 1.1", "M 1.2", "M 1.3"]),
    ("M2", "M2. Gini", ["M 2.1", "M 2.2", "M 2.3", "M 2.4"]),
    ("M3", "M3. Multicollinearity", ["M 3.1", "M 3.2"]),
    ("M4", "M4. Calibration", ["M 4.1", "M 4.2", "M 4.3"]),
    ("M5", "M5. Stability", ["M 5.1", "M 5.2", "M 5.3", "M 5.4"]),
]

inline_tests = {}
inline_results = {}

def _display_test_result(result: dict, key: str | None = None, max_items: int | None = None) -> None:
    if not isinstance(result, dict):
        display(HTML(f"<p>Unexpected result type: {escape(str(type(result)))}</p>"))
        return

    if key and key in result:
        display(HTML(str(result[key])))
        return
    if "combined" in result:
        display(HTML(str(result["combined"])))
        return
    if "DASHBOARD" in result:
        display(HTML(str(result["DASHBOARD"])))
        return

    shown = 0
    for subkey, html in result.items():
        display(HTML(f"<h4 style='margin:8px 0'>{escape(str(subkey))}</h4>" + str(html)))
        shown += 1
        if max_items and shown >= max_items:
            break

def run_inline_test(code: str, key: str | None = None, max_items: int | None = None, display_result: bool = True) -> dict:
    test = TEST_FACTORIES[code]()
    params = TEST_PARAMS[code]
    print(f"Running {code} {test.test_name}...")

    try:
        result = test.run(**params)
    except Exception as exc:
        test.test_signal = "red"
        result = {
            "DASHBOARD": (
                f"<h4>{escape(code)} {escape(test.test_name)}</h4>"
                f"<p style='color:red'><b>Error:</b> {escape(repr(exc))}</p>"
            )
        }

    inline_tests[code] = test
    inline_results[code] = result
    print(f"Signal: {test.test_signal}")

    if display_result:
        _display_test_result(result, key=key, max_items=max_items)
    return result


## M0: Data Quality


### M 0.1: Data Quality Overview


In [ ]:
run_inline_test("M 0.1")

### M 0.4: Monthly Distributions


In [ ]:
run_inline_test("M 0.4")

### M 0.5: Mode and Missing by Period


In [ ]:
run_inline_test("M 0.5")

## M1: Distribution


### M 1.1: TRAIN vs GENPOP PSI


In [ ]:
run_inline_test("M 1.1")

### M 1.2: Validation vs GENPOP PSI


In [ ]:
run_inline_test("M 1.2")

### M 1.3: Max Date Recency


In [ ]:
run_inline_test("M 1.3")

## M2: Gini


### M 2.1: Gini Test


In [ ]:
run_inline_test("M 2.1")

### M 2.2: PowerStat by Factors


In [ ]:
run_inline_test("M 2.2")

### M 2.3: Gini Dynamics


In [ ]:
run_inline_test("M 2.3")

### M 2.4: PowerStat Uplift


In [ ]:
run_inline_test("M 2.4")

## M3: Multicollinearity


### M 3.1: Likelihood-ratio tests


In [ ]:
run_inline_test("M 3.1")

### M 3.2: Multicollinearity Test


In [ ]:
run_inline_test("M 3.2")

## M4: Calibration


### M 4.1: Predicted vs Observed Rate


In [ ]:
run_inline_test("M 4.1")

### M 4.2: Calibration by Prediction Bins


In [ ]:
run_inline_test("M 4.2")

### M 4.3: Calibration by Dates


In [ ]:
run_inline_test("M 4.3")

## M5: Stability


### M 5.1: Model Gini Stability


In [ ]:
run_inline_test("M 5.1")

### M 5.2: Factor Gini Stability


In [ ]:
run_inline_test("M 5.2")

### M 5.3: Score PSI Stability


In [ ]:
run_inline_test("M 5.3")

### M 5.4: Factors PSI Stability


In [ ]:
run_inline_test("M 5.4")

##  сборка HTML-отчета



In [ ]:
RUN_MISSING_TESTS_FOR_REPORT = True

missing_codes = [
    code
    for _, _, codes in TEST_GROUPS
    for code in codes
    if code not in inline_results
]

if missing_codes and not RUN_MISSING_TESTS_FOR_REPORT:
    raise RuntimeError(f"Перед сборкой отчета запустите пропущенные тесты: {missing_codes}")

for code in missing_codes:
    run_inline_test(code, display_result=False)

report = ModelReportBuilder()
for group_code, group_name, codes in TEST_GROUPS:
    group = TestGroupBuilder(group_code=group_code, group_name=group_name)
    for code in codes:
        group.add_test(inline_tests[code])
        group.test_results[code] = inline_results[code]
    report.add_group(group)

html = report.generate_html()
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = REPORTS_DIR / f"OMTPL_Retail_Classical_FREQ_inline_{datetime.now():%d_%m_%Y}.html"
report_path.write_text(html, encoding="utf-8")

print(f"Report saved: {report_path}")
print(f"Report size: {report_path.stat().st_size / 1024 / 1024:.2f} MB")
display(FileLink(str(report_path)))